<a href="https://colab.research.google.com/github/Khalidsyfullah/USplitVQA/blob/main/Direct_Model_Apply/New_BioMedClip_Centralized_VIZWIZ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
================================================================================
CENTRALIZED BiomedCLIP VizWiz — Colab L4 Notebook
================================================================================
BiomedCLIP ViT-B/16 + PubMedBERT → 4-layer BiDir Cross-Attn → Query Pool → MLP
Dataset: VizWiz-VQA (val split → 80/20, majority vote, vocab cap 300)
Results → vizwiz_centralized_biomedclip_results.xlsx
================================================================================
"""
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch", "transformers", "datasets", "openpyxl", "tqdm"])
import re
import os, random, time
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR="/content/results"; os.makedirs(OUTPUT_DIR,exist_ok=True)

D=768; HEADS=8; FUSE_LAYERS=2; DROP=0.25; EPOCHS=30; BS=32; LR=2e-5; WD=1e-4; LS=0.1
FREEZE_EP=5; ENC_SCALE=0.1; ES_PAT=8; LR_PAT=4; LR_FAC=0.5; MIN_LR=1e-7; VAL_SPLIT=0.15
MAX_ANS_VOCAB=300; MIN_ANS_FREQ=3

# ─────────────────────────────────────────────────────────────────────
# 4. VizWiz  (e.g., Eldon/VizWiz or any HF mirror)
# ─────────────────────────────────────────────────────────────────────
# ~31,000 QA pairs from blind users · real-world photos
# Highly noisy: unanswerable questions, poor image quality,
# answers from 10 crowd annotators. Very diverse open-ended answers
# (objects, colors, text reading, brands, counts, etc.)

def normalize_answer_vizwiz(ans: str) -> str:
    """Normalize VizWiz answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Unanswerable / unsuitable canonicalization ──
    unanswerable_set = {
        'unanswerable', 'unsuitable', 'unsuitable image',
        'not answerable', 'cant answer', 'cannot answer',
        'i dont know', 'i don\'t know', 'i do not know',
        'unable to answer', 'not sure', 'unclear',
        'unreadable', 'cannot be determined', 'cant be determined',
        'can not be determined', 'not clear', 'blurry',
        'too blurry', 'too dark', 'no answer', 'na', 'n/a',
        'cannot tell', 'cant tell', 'hard to tell',
        'impossible to tell', 'nothing', 'not possible',
    }
    if ans in unanswerable_set:
        return 'unanswerable'

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true',
               'yes it is', 'yes, it is', 'yea', 'ya'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'no it is not', 'nah'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric canonicalization ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10',
                   'eleven': '11', 'twelve': '12', 'thirteen': '13',
                   'fourteen': '14', 'fifteen': '15', 'twenty': '20',
                   'thirty': '30', 'forty': '40', 'fifty': '50',
                   'hundred': '100'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Color normalization (very common in VizWiz) ──
    color_map = {
        'blue': 'blue', 'light blue': 'blue', 'dark blue': 'blue',
        'navy': 'blue', 'navy blue': 'blue', 'royal blue': 'blue',
        'red': 'red', 'dark red': 'red', 'light red': 'red',
        'maroon': 'red', 'crimson': 'red', 'burgundy': 'red',
        'green': 'green', 'light green': 'green', 'dark green': 'green',
        'lime': 'green', 'olive': 'green',
        'yellow': 'yellow', 'light yellow': 'yellow', 'gold': 'yellow',
        'golden': 'yellow',
        'orange': 'orange',
        'pink': 'pink', 'light pink': 'pink', 'hot pink': 'pink',
        'magenta': 'pink',
        'purple': 'purple', 'violet': 'purple', 'lavender': 'purple',
        'brown': 'brown', 'tan': 'brown', 'beige': 'brown',
        'khaki': 'brown',
        'white': 'white', 'off white': 'white', 'cream': 'white',
        'ivory': 'white',
        'black': 'black', 'dark': 'black',
        'gray': 'gray', 'grey': 'gray', 'silver': 'gray',
        'light gray': 'gray', 'light grey': 'gray',
        'dark gray': 'gray', 'dark grey': 'gray',
    }
    if ans in color_map:
        return color_map[ans]

    # ── Common object synonyms ──
    object_map = {
        'cellphone': 'phone', 'cell phone': 'phone', 'mobile': 'phone',
        'mobile phone': 'phone', 'smartphone': 'phone', 'iphone': 'phone',
        'tv': 'television', 'television': 'television',
        'laptop': 'laptop', 'computer': 'laptop', 'notebook': 'laptop',
        'can': 'can', 'cans': 'can', 'tin': 'can',
        'bottle': 'bottle', 'bottles': 'bottle',
        'box': 'box', 'boxes': 'box', 'package': 'box',
        'shirt': 'shirt', 'tshirt': 'shirt', 't-shirt': 'shirt',
        't shirt': 'shirt', 'tee shirt': 'shirt',
        'pants': 'pants', 'trousers': 'pants', 'jeans': 'pants',
        'shoe': 'shoe', 'shoes': 'shoe', 'sneaker': 'shoe',
        'sneakers': 'shoe',
        'remote': 'remote', 'remote control': 'remote',
        'glasses': 'glasses', 'eyeglasses': 'glasses',
        'sunglasses': 'sunglasses',
        'soda': 'soda', 'pop': 'soda', 'soft drink': 'soda',
        'coke': 'coca cola', 'coca-cola': 'coca cola',
        'pepsi': 'pepsi', 'dr pepper': 'dr pepper',
        'cat': 'cat', 'cats': 'cat', 'kitten': 'cat',
        'dog': 'dog', 'dogs': 'dog', 'puppy': 'dog',
        'dollar': 'dollar', 'dollars': 'dollar',
        'cent': 'cent', 'cents': 'cent',
    }
    if ans in object_map:
        return object_map[ans]

    # ── Remove articles and fillers ──
    ans = re.sub(r'^(the|a|an|its|it is|this is|that is|it\'s|i think)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Remove trailing period ──
    ans = ans.rstrip('.')

    return ans

# ── LOAD VizWiz ──
print("\n"+"="*60+"\nLOADING VizWiz INTO RAM\n"+"="*60)
from datasets import load_dataset; ds=load_dataset('lmms-lab/VizWiz-VQA')
def majority_answer(answers):
    if not answers: return ''
    clean=[str(a.get('answer','') if isinstance(a,dict) else a).strip().lower() for a in answers]
    clean=[a for a in clean if a and a not in ('unanswerable','unsuitable')]
    return Counter(clean).most_common(1)[0][0] if clean else ''
all_samples=[]
for s in tqdm(ds['val'],desc="VizWiz"):
    try:
        img=s.get('image'); q=str(s.get('question',''))
        if not img or not q: continue
        answer=s.get('answer')
        if answer is not None: answer=str(answer).strip().lower()
        else: answer=majority_answer(s.get('answers',[]))
        answer = normalize_answer_vizwiz(answer)
        if not answer or answer in ('unanswerable','unsuitable',''): continue
        all_samples.append({'image':img.convert('RGB'),'question':q,'answer':answer})
    except: continue
del ds; random.shuffle(all_samples); sp=int(len(all_samples)*0.8)
train_samples,test_samples=all_samples[:sp],all_samples[sp:]
all_ans=[s['answer'] for s in all_samples]; counts=Counter(all_ans)
filtered=[(a,c) for a,c in counts.most_common() if c>=MIN_ANS_FREQ][:MAX_ANS_VOCAB]
answer_vocab={'<unk>':0}
for i,(a,_) in enumerate(sorted(filtered,key=lambda x:x[0])): answer_vocab[a]=i+1
num_classes=len(answer_vocab)
indices=list(range(len(train_samples))); random.shuffle(indices); n_val=int(len(indices)*VAL_SPLIT)
trn=[train_samples[i] for i in indices[n_val:]]; val=[train_samples[i] for i in indices[:n_val]]
print(f"  Train:{len(trn)}, Val:{len(val)}, Test:{len(test_samples)}, Classes:{num_classes}")

# ── BiomedCLIP ──
print("\n"+"="*60+"\nLOADING BiomedCLIP\n"+"="*60)
from open_clip import create_model_and_transforms, get_tokenizer
CN="hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
clip_model, preprocess_train, preprocess_val = create_model_and_transforms(CN)
tokenizer = get_tokenizer(CN); clip_model=clip_model.to(device)

class VQADataset(Dataset):
    def __init__(s,samples,vocab,pp,tok): s.samples=samples; s.vocab=vocab; s.pp=pp; s.tok=tok; s.unk=vocab.get('<unk>',0)
    def __len__(s): return len(s.samples)
    def __getitem__(s,i): x=s.samples[i]; return s.pp(x['image']),s.tok([x['question']])[0],s.vocab.get(x['answer'],s.unk)
def collate_fn(b):
    imgs,txts,lbls=zip(*b); imgs=torch.stack(imgs); mx=max(t.shape[0] for t in txts)
    p=torch.zeros(len(txts),mx,dtype=txts[0].dtype)
    for i,t in enumerate(txts): p[i,:t.shape[0]]=t
    return imgs,p,torch.tensor(lbls,dtype=torch.long)
train_loader=DataLoader(VQADataset(trn,answer_vocab,preprocess_train,tokenizer),batch_size=BS,shuffle=True,num_workers=2,pin_memory=True,collate_fn=collate_fn)
val_loader=DataLoader(VQADataset(val,answer_vocab,preprocess_val,tokenizer),batch_size=BS,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_fn)
test_loader=DataLoader(VQADataset(test_samples,answer_vocab,preprocess_val,tokenizer),batch_size=BS,shuffle=False,num_workers=2,pin_memory=True,collate_fn=collate_fn)

# ── MODEL ──
class FusionTransformerLayer(nn.Module):
    def __init__(s,dim,nh,do=0.1):
        super().__init__(); s.v2t_a=nn.MultiheadAttention(dim,nh,dropout=do,batch_first=True); s.v2t_n1=nn.LayerNorm(dim); s.v2t_n2=nn.LayerNorm(dim)
        s.v2t_f=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(do),nn.Linear(dim*4,dim),nn.Dropout(do))
        s.t2v_a=nn.MultiheadAttention(dim,nh,dropout=do,batch_first=True); s.t2v_n1=nn.LayerNorm(dim); s.t2v_n2=nn.LayerNorm(dim)
        s.t2v_f=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(do),nn.Linear(dim*4,dim),nn.Dropout(do))
    def forward(s,v,t,kpm=None):
        o,_=s.v2t_a(v,t,t,key_padding_mask=kpm); v=s.v2t_n1(v+o); v=s.v2t_n2(v+s.v2t_f(v))
        o,_=s.t2v_a(t,v,v); t=s.t2v_n1(t+o); t=s.t2v_n2(t+s.t2v_f(t)); return v,t

class BiomedCLIPVQAModel(nn.Module):
    def __init__(s,clip,nc):
        super().__init__(); s.clip=clip; s.vp=nn.Identity(); s.tp=nn.Identity()
        s.fl=nn.ModuleList([FusionTransformerLayer(D,HEADS,DROP) for _ in range(FUSE_LAYERS)])
        s.pq=nn.Parameter(torch.randn(1,1,D)*0.02); s.pa=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.pn=nn.LayerNorm(D)
        s.head=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(DROP),nn.Linear(D,D//2),nn.GELU(),nn.Dropout(DROP),nn.Linear(D//2,nc))
    def ei(s,img):
        ve=s.clip.visual; x=ve.trunk.patch_embed(img); x=ve.trunk._pos_embed(x); x=ve.trunk.patch_drop(x); x=ve.trunk.norm_pre(x); x=ve.trunk.blocks(x); return ve.trunk.norm(x)
    def et(s,ids): te=s.clip.text; am=(ids!=0).long(); return te.transformer(input_ids=ids,attention_mask=am).last_hidden_state, am
    def forward(s,img,ids):
        v=s.vp(s.ei(img)); t,am=s.et(ids); t=s.tp(t); kpm=(am==0)
        for f in s.fl: v,t=f(v,t,kpm=kpm)
        c=torch.cat([v,t],1); B=c.shape[0]; pq=s.pq.expand(B,-1,-1); p,_=s.pa(pq,c,c); return s.head(s.pn(pq+p).squeeze(1))

model=BiomedCLIPVQAModel(clip_model,num_classes).to(device)
nt=sum(p.numel() for p in model.parameters()); ne=sum(p.numel() for p in model.clip.parameters()); nfh=nt-ne
print(f"\n  Total:{nt:,}, Encoder:{ne:,}, Fusion+Head:{nfh:,}")

# ── TRAINING ──
print("\n"+"="*60+"\nTRAINING (Centralized)\n"+"="*60)
def get_groups(frozen):
    enc=list(model.clip.parameters()); enc_ids=set(id(p) for p in enc)
    fh=[p for p in model.parameters() if id(p) not in enc_ids and p.requires_grad]
    if frozen:
        for p in enc: p.requires_grad=False
        return [{'params':fh,'lr':LR}]
    for p in enc: p.requires_grad=True
    return [{'params':[p for p in enc if p.requires_grad],'lr':LR*ENC_SCALE},{'params':fh,'lr':LR}]

criterion=nn.CrossEntropyLoss(label_smoothing=LS); scaler=torch.amp.GradScaler('cuda') if device.type=='cuda' else None
optimizer=torch.optim.AdamW(get_groups(True),weight_decay=WD)
history={'epoch':[],'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],'test_loss':[],'test_acc':[],'lr':[],'epoch_time':[]}
best_val,best_state,pat,lr_pat=0.0,None,0,0

@torch.no_grad()
def ev(loader):
    model.eval(); ls,c,t=0.0,0,0
    for i,d,l in loader: i,d,l=i.to(device),d.to(device),l.to(device); lo=model(i,d); loss=criterion(lo,l); ls+=loss.item()*l.size(0); c+=(lo.argmax(-1)==l).sum().item(); t+=l.size(0)
    return ls/t,100*c/t

for epoch in range(1,EPOCHS+1):
    t0=time.time()
    if epoch==FREEZE_EP+1: print(f"\n  === Unfreezing ==="); optimizer=torch.optim.AdamW(get_groups(False),weight_decay=WD)
    model.train(); tl,tc,tt=0.0,0,0
    pb=tqdm(train_loader,desc=f"E{epoch:02d}/{EPOCHS}",leave=False)
    for i,d,l in pb:
        i,d,l=i.to(device),d.to(device),l.to(device); optimizer.zero_grad()
        with torch.amp.autocast('cuda',enabled=scaler is not None): lo=model(i,d); loss=criterion(lo,l)
        if scaler: scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
        else: loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        tl+=loss.item()*l.size(0); tc+=(lo.argmax(-1)==l).sum().item(); tt+=l.size(0); pb.set_postfix(l=f"{loss.item():.3f}",a=f"{100*tc/tt:.1f}%")
    tl/=tt; ta=100*tc/tt; vl,va=ev(val_loader); tel,tea=ev(test_loader); lr=optimizer.param_groups[-1]['lr']; et=time.time()-t0
    history['epoch'].append(epoch); history['train_loss'].append(tl); history['train_acc'].append(ta); history['val_loss'].append(vl); history['val_acc'].append(va)
    history['test_loss'].append(tel); history['test_acc'].append(tea); history['lr'].append(lr); history['epoch_time'].append(et)
    mk=""
    if va>best_val: best_val=va; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; pat=lr_pat=0; mk=" ★"
    else: pat+=1; lr_pat+=1
    print(f"E{epoch:02d} [{et:.1f}s]  Train:{tl:.4f}/{ta:.1f}%  Val:{vl:.4f}/{va:.1f}%  Test:{tea:.1f}%{mk}")
    if lr_pat>=LR_PAT:
        for pg in optimizer.param_groups: pg['lr']=max(pg['lr']*LR_FAC,MIN_LR); lr_pat=0
    if pat>=ES_PAT: print("  Early stop"); break
if best_state: model.load_state_dict(best_state)
tel,tea=ev(test_loader); print(f"\n{'='*60}\nFINAL: {tea:.2f}%\n{'='*60}")

# ── EXCEL ──
import openpyxl; from openpyxl.styles import Font,PatternFill,Alignment
wb=openpyxl.Workbook(); ws=wb.active; ws.title="Training"
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF'); hfi=PatternFill(start_color='1A5276',end_color='1A5276',fill_type='solid')
headers=['Epoch','Train Loss','Train Acc (%)','Val Loss','Val Acc (%)','Test Loss','Test Acc (%)','LR','Time (s)']
for c,h in enumerate(headers,1): cl=ws.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
for i,ep in enumerate(history['epoch']):
    r=i+2
    for c,k in enumerate(['epoch','train_loss','train_acc','val_loss','val_acc','test_loss','test_acc','lr','epoch_time'],1):
        v=history[k][i]; ws.cell(row=r,column=c,value=round(v,4) if isinstance(v,float) else v)
ws2=wb.create_sheet("Summary")
for i,(k,v) in enumerate([("Method","Centralized BiomedCLIP"),("Dataset","VizWiz"),("Total",f"{nt:,}"),("Classes",num_classes),("Best Val",round(best_val,2)),("Final Test",round(tea,2))],1):
    ws2.cell(row=i,column=1,value=k).font=Font(bold=True,name='Arial'); ws2.cell(row=i,column=2,value=v)
for s in [ws,ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2
p=f"{OUTPUT_DIR}/vizwiz_centralized_biomedclip_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")